# Fine-tune a Pretrained `GLACIER` Model

Use this notebook to fine-tune a pretrained `GLACIER` model on the ESOL dataset.


In [ ]:
import os
import sys

import lightning.pytorch as pl
import torch
import torch.nn.functional as F
from huggingface_hub import snapshot_download
from torchmetrics import MeanSquaredError
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from torch import nn

repo_dir = snapshot_download(repo_id="glacier-hf/GLACIER-100k-MiniMol")
sys.path.append(repo_dir)


## ESOL Regression Example

https://ogb.stanford.edu/docs/graphprop/

Note: This dataset has been preprocessed (as described in the paper) and is ML-ready. 


## Define dataset file paths


In [ ]:
train_path = os.path.join(".", "example_data", "train_ESOL.tab")
valid_path = os.path.join(".", "example_data", "valid_ESOL.tab")
test_path = os.path.join(".", "example_data", "test_ESOL.tab")
checkpoint_dir = os.path.join(".", "checkpoints", "esol_finetune")


## Load data


In [ ]:
from data.utils import load_data

train = load_data(train_path)
valid = load_data(valid_path)
test = load_data(test_path)

train_smis, train_labels = train["smiles"].values, train["Y"].values
valid_smis, valid_labels = valid["smiles"].values, valid["Y"].values
test_smis, test_labels = test["smiles"].values, test["Y"].values


## Create dataloaders


In [ ]:
from data.dataloader import SmilesMoleculeDataset, build_dataloader

train_dataset = SmilesMoleculeDataset(train_smis, labels=train_labels, is_train=True) 
train_dataloader = build_dataloader(train_dataset, batch_size=64, num_workers=0) 

valid_dataset = SmilesMoleculeDataset(valid_smis, labels=valid_labels)
valid_dataloader = build_dataloader(valid_dataset, batch_size=64, num_workers=0) 

test_dataset = SmilesMoleculeDataset(test_smis, labels=test_labels)
test_dataloader = build_dataloader(test_dataset, batch_size=64, num_workers=0) 


## Load pretrained `GLACIER`


In [ ]:
from glacier_student import Glacier

glacier = Glacier.from_pretrained("glacier-hf/GLACIER-100k-MiniMol")


## Fine-tune and save checkpoints


In [ ]:
EMBED_DIM = 512


class GlacierFinetuneModule(pl.LightningModule):
    def __init__(self, backbone, text_lr=3e-5, backbone_lr=1e-4, head_lr=1e-3, weight_decay=1e-2):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(EMBED_DIM, 1)
        self.text_lr = text_lr
        self.backbone_lr = backbone_lr
        self.head_lr = head_lr
        self.weight_decay = weight_decay

        # RMSE 
        self.test_rmse = MeanSquaredError(squared=False)

    def forward(self, batch):
        return self.head(self.backbone(batch))

    def _step(self, batch, stage):
        preds = self(batch)
        y = batch["Y"].to(self.device)
        if y.ndim == 1:
            y = y.unsqueeze(-1)
        loss = F.mse_loss(preds, y)
        self.log(f"{stage}_loss", loss, prog_bar=(stage == "val"), on_epoch=True, batch_size=y.shape[0])
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")

    def test_step(self, batch, batch_idx):
            preds = self(batch)
            y = batch["Y"]
            if y.ndim == 1:
                y = y.unsqueeze(-1)

            # Compute RMSE loss for logging
            loss = F.mse_loss(preds, y)
            self.log("test_loss", loss, on_epoch=True, batch_size=y.shape[0])
            self.test_rmse(preds, y)
            self.log("test_rmse", self.test_rmse, on_epoch=True, prog_bar=True, batch_size=y.shape[0])
            return loss

    def configure_optimizers(self):
        text_params = list(self.backbone.text_encoder.parameters())
        text_ids = {id(p) for p in text_params}
        backbone_params = [p for p in self.backbone.parameters() if id(p) not in text_ids]
        return torch.optim.AdamW(
            [
                {"params": backbone_params, "lr": self.backbone_lr},
                {"params": text_params, "lr": self.text_lr},
                {"params": self.head.parameters(), "lr": self.head_lr},
            ],
            weight_decay=self.weight_decay,
        )


module = GlacierFinetuneModule(glacier)
checkpoint_cb = ModelCheckpoint(
    dirpath=checkpoint_dir,
    filename="best",
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    save_last=True,
)
trainer = pl.Trainer(
    max_epochs=50,
    callbacks=[checkpoint_cb, EarlyStopping(monitor="val_loss", mode="min", patience=10)],
    accelerator="auto",
    devices=1,
    default_root_dir=checkpoint_dir,
    enable_progress_bar=True,
)
trainer.fit(module, train_dataloader, valid_dataloader)
print(f"Best checkpoint: {checkpoint_cb.best_model_path}")

# Evaluate

In [ ]:
test_results = trainer.test(
    model=module,
    dataloaders=test_dataloader,
    ckpt_path=checkpoint_cb.best_model_path 
)